# VCT: YouTube → cloud processing → CSV
Run setup, configure a URL and run extraction. No local video upload is needed.
OCR works immediately; custom YOLO weights and an alias catalog improve proposals.
All outputs need review. Training requires labeled frames and is a separate optional step.
Enable a GPU. Colab/Kaggle resources and YouTube downloads are not guaranteed.


In [ ]:
import sys, subprocess, json
from pathlib import Path

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "-q",
        "yt-dlp[default]>=2026.1",
        "opencv-python-headless>=4.10,<5",
        "easyocr>=1.7,<2",
        "ultralytics>=8.3,<9",
    ],
    check=True,
)
# ffmpeg is required by the downloader for merging video and audio.
import shutil

if shutil.which("ffmpeg") is None:
    raise RuntimeError("Install ffmpeg in this runtime before downloading.")
modules = {
    "acquisition.py": '"""Supported JS runtime setup and bounded, explicit YouTube authentication options."""\n\nimport hashlib\nimport io\nimport json\nimport os\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport urllib.request\nfrom pathlib import Path\nfrom zipfile import ZipFile\n\n\ndef detect_runtime():\n    for name, minimum in [("deno", (2, 3, 0)), ("node", (22, 0, 0))]:\n        executable = shutil.which(name)\n        if not executable:\n            continue\n        try:\n            version = subprocess.check_output([executable, "--version"], text=True, timeout=10)\n            match = re.search(r"(\\d+)\\.(\\d+)\\.(\\d+)", version)\n            if match and tuple(map(int, match.groups())) >= minimum:\n                return {name: {"path": executable}}\n        except (OSError, subprocess.SubprocessError):\n            continue\n    return {}\n\n\ndef ensure_runtime():\n    """Install the official Deno Linux binary in the notebook, checking its asset digest."""\n    found = detect_runtime()\n    if found:\n        return found\n    if platform.system() != "Linux" or platform.machine() not in {"x86_64", "AMD64"}:\n        raise RuntimeError("Install Deno >=2.3 or Node >=22 and rerun setup")\n    request = urllib.request.Request(\n        "https://api.github.com/repos/denoland/deno/releases/latest",\n        headers={"User-Agent": "ValoMind-notebook"},\n    )\n    with urllib.request.urlopen(request, timeout=30) as response:\n        release = json.load(response)\n    asset = next(\n        (item for item in release["assets"] if item["name"] == "deno-x86_64-unknown-linux-gnu.zip"),\n        None,\n    )\n    if not asset or not str(asset.get("digest", "")).startswith("sha256:"):\n        raise RuntimeError("Official Deno asset has no verification digest; install Deno manually")\n    url = asset["browser_download_url"]\n    if not url.startswith("https://github.com/denoland/deno/releases/download/"):\n        raise RuntimeError("Unexpected Deno release asset location")\n    with urllib.request.urlopen(url, timeout=120) as response:\n        data = response.read()\n    if hashlib.sha256(data).hexdigest() != asset["digest"].split(":", 1)[1]:\n        raise RuntimeError("Deno asset checksum mismatch")\n    target = Path.home() / ".local/bin/deno"\n    target.parent.mkdir(parents=True, exist_ok=True)\n    with ZipFile(io.BytesIO(data)) as archive:\n        binary = archive.read("deno")\n    temporary = target.with_suffix(".tmp")\n    temporary.write_bytes(binary)\n    temporary.chmod(0o755)\n    temporary.replace(target)\n    os.environ["PATH"] = str(target.parent) + os.pathsep + os.environ.get("PATH", "")\n    found = detect_runtime()\n    if not found:\n        raise RuntimeError("Deno installation did not pass the runtime version check")\n    return found\n\n\ndef youtube_options(cookie_file=None, js_runtime=None, export_root=None):\n    options = {"js_runtimes": js_runtime or detect_runtime()}\n    if cookie_file:\n        path = Path(cookie_file).expanduser().resolve()\n        if not path.is_file():\n            raise ValueError("COOKIE_FILE does not exist in this cloud runtime")\n        if export_root and path.is_relative_to(Path(export_root).resolve()):\n            raise ValueError(\n                "Keep COOKIE_FILE outside WORK_ROOT so it cannot enter result archives"\n            )\n        options["cookiefile"] = str(path)\n    return options\n\n\ndef download_error_message(error):\n    message = str(error).casefold()\n    if "not a bot" in message or "sign in to confirm" in message:\n        return (\n            "YouTube requires sign-in/verification from this cloud runtime. The JS runtime "\n            "alone cannot guarantee access. Stop retrying; optionally configure COOKIE_FILE "\n            "from your own authorized browser session outside WORK_ROOT, or reuse the "\n            "previously downloaded video in persistent cloud storage. Never paste cookies "\n            "into chat or commit them. No automatic proxy/account rotation is attempted."\n        )\n    return "YouTube acquisition failed. Check video availability, runtime setup and the downloader logs."\n\n\ndef cached_video(folder, key):\n    """Reuse only a decodable completed video with matching stored source metadata."""\n    info = folder / "video.info.json"\n    if not info.is_file():\n        return None\n    try:\n        metadata = json.loads(info.read_text(encoding="utf-8"))\n        if metadata.get("id") != key or not shutil.which("ffprobe"):\n            return None\n        for suffix in [".mp4", ".webm", ".mkv"]:\n            video = folder / ("video" + suffix)\n            if not video.is_file() or video.stat().st_size == 0:\n                continue\n            probe = subprocess.check_output(\n                [\n                    "ffprobe",\n                    "-v",\n                    "error",\n                    "-show_entries",\n                    "format=duration",\n                    "-of",\n                    "json",\n                    str(video),\n                ],\n                text=True,\n                timeout=30,\n            )\n            duration = float(json.loads(probe)["format"]["duration"])\n            expected = float(metadata.get("duration") or 0)\n            if duration > 0 and (\n                not expected or abs(duration - expected) <= max(2, expected * 0.01)\n            ):\n                return video\n    except (OSError, ValueError, KeyError, subprocess.SubprocessError):\n        return None\n    return None\n\n\ndef restore_source_archive(archive_path, root, key):\n    """Restore only the selected prior source video and metadata, never arbitrary ZIP paths."""\n    if not re.fullmatch(r"[A-Za-z0-9_-]{11}", key):\n        raise ValueError("Invalid source video ID")\n    folder = Path(root) / "sources" / key\n    existing = cached_video(folder, key)\n    if existing:\n        return existing\n    prefix = f"sources/{key}/"\n    with ZipFile(archive_path) as archive:\n        names = set(archive.namelist())\n        info_name = prefix + "video.info.json"\n        video_name = next(\n            (\n                prefix + "video" + suffix\n                for suffix in [".mp4", ".webm", ".mkv"]\n                if prefix + "video" + suffix in names\n            ),\n            None,\n        )\n        if not video_name or info_name not in names:\n            raise ValueError(\n                "This archive has no full source video. Use the FIRST valomind_results.zip, not the later metadata-only exports."\n            )\n        if archive.getinfo(info_name).file_size > 5_000_000:\n            raise ValueError("Source metadata is unexpectedly large")\n        metadata_bytes = archive.read(info_name)\n        metadata = json.loads(metadata_bytes)\n        if metadata.get("id") != key:\n            raise ValueError("Archive metadata does not match the requested YouTube video")\n        expected = set()\n        for name in names:\n            if name.startswith("jobs/") and name.endswith("/manifest.json"):\n                if archive.getinfo(name).file_size > 1_000_000:\n                    raise ValueError("Unexpectedly large job manifest")\n                manifest = json.loads(archive.read(name))\n                digest = manifest.get("source_sha256", "")\n                if manifest.get("video_id") == key and re.fullmatch(r"[a-f0-9]{64}", digest):\n                    expected.add(digest)\n        folder.mkdir(parents=True, exist_ok=True)\n        destination = folder / Path(video_name).name\n        temporary = folder / (destination.name + ".restore.part")\n        digest = hashlib.sha256()\n        try:\n            with archive.open(video_name) as source, temporary.open("wb") as output:\n                for chunk in iter(lambda: source.read(4 * 1024 * 1024), b""):\n                    digest.update(chunk)\n                    output.write(chunk)\n            if temporary.stat().st_size == 0 or (expected and digest.hexdigest() not in expected):\n                raise ValueError("Restored source checksum does not match the archived job")\n            temporary.replace(destination)\n            info = folder / "video.info.json"\n            info_temp = folder / "video.info.json.restore.part"\n            info_temp.write_bytes(metadata_bytes)\n            info_temp.replace(info)\n        finally:\n            if temporary.exists():\n                temporary.unlink()\n    restored = cached_video(folder, key)\n    if not restored:\n        raise ValueError(\n            "Restored video failed ffprobe validation; check ffmpeg/ffprobe setup and archive completeness"\n        )\n    print(\n        "Restored prior source into the cloud cache; YouTube access is not needed for this video."\n    )\n    return restored\n',
    "entities.py": '"""Conservative entity proposals. Co-occurrence is never a player-agent assignment."""\n\nimport re\nimport unicodedata\n\nKINDS = {"player", "team", "map", "character_agent"}\n\n\ndef normalize(value):\n    return " ".join(re.findall(r"\\w+", unicodedata.normalize("NFKC", value).casefold()))\n\n\ndef validate_catalog(catalog):\n    if not isinstance(catalog, list):\n        raise TypeError("Catalog must be a list")\n    seen = set()\n    for entry in catalog:\n        if set(entry) != {"kind", "id", "aliases"} or entry["kind"] not in KINDS:\n            raise ValueError("Catalog entries require kind, id and aliases")\n        if not isinstance(entry["id"], str) or not entry["id"].strip():\n            raise ValueError("Entity ID must be a nonempty string")\n        if (entry["kind"], entry["id"]) in seen:\n            raise ValueError("Duplicate entity ID")\n        seen.add((entry["kind"], entry["id"]))\n        if not isinstance(entry["aliases"], list) or not entry["aliases"]:\n            raise ValueError("Aliases must be a nonempty list")\n        if any(not isinstance(alias, str) or not normalize(alias) for alias in entry["aliases"]):\n            raise ValueError("Aliases must contain readable text")\n    return catalog\n\n\ndef resolve(proposals, catalog):\n    """Match complete OCR strings or explicit detector labels; reject ambiguous aliases."""\n    index = {}\n    for entry in validate_catalog(catalog):\n        for alias in entry["aliases"]:\n            index.setdefault(normalize(alias), set()).add((entry["kind"], entry["id"]))\n    result = []\n    for proposal in proposals:\n        text = proposal.get("text", "") if proposal["kind"] == "ocr" else proposal["label"]\n        matches = index.get(normalize(text), set())\n        if len(matches) != 1:\n            continue\n        kind, identity = next(iter(matches))\n        entity = dict(proposal, kind=f"entity.{kind}", label=identity)\n        entity[\n            {\n                "player": "player_id",\n                "team": "team_id",\n                "map": "map_id",\n                "character_agent": "character_agent_id",\n            }[kind]\n        ] = identity\n        result.append(entity)\n    return result\n',
    "ocr_retry.py": '"""Retry only unresolved roster fields, preserving all original OCR evidence."""\n\nfrom collections import Counter\n\nfrom .roster import parse_frame\n\nRETRY_VERSION = "crop_retry_v1"\n\n\ndef retry_regions(rows, width, height, catalog=None):\n    roster = parse_frame(rows, width, height, catalog)\n    regions = []\n    for side, box in [("left", (710, 0, 795, 55)), ("right", (1125, 0, 1200, 55))]:\n        if not any(row["side"] == side and row["team_id"] for row in roster):\n            regions.append(("team", box))\n    layouts = Counter(row["layout"] for row in roster)\n    # Do not select a layout when both plausible layouts have recognized names.\n    if len(layouts) == 1 and next(iter(layouts.values())) >= 3:\n        layout = next(iter(layouts))\n        existing = {(row["side"], row["slot"]) for row in roster}\n        for side in ["left", "right"]:\n            for slot in range(1, 6):\n                if (side, slot) in existing:\n                    continue\n                if layout == "buy":\n                    y = 783 + (slot - 1) * 56\n                    box = (645, y, 815, y + 53) if side == "left" else (1165, y, 1275, y + 53)\n                else:\n                    y = 550 + (slot - 1) * 106\n                    box = (77, y, 255, y + 45) if side == "left" else (1725, y, 1841, y + 45)\n                regions.append(("player_name", box))\n    return [\n        (\n            kind,\n            tuple(\n                round(value * (width / 1920 if i % 2 == 0 else height / 1080))\n                for i, value in enumerate(box)\n            ),\n        )\n        for kind, box in regions\n    ]\n\n\ndef retry(frame, reader, rows, catalog=None):\n    import cv2\n\n    height, width = frame.shape[:2]\n    output = []\n    for kind, (x1, y1, x2, y2) in retry_regions(rows, width, height, catalog):\n        crop = frame[y1:y2, x1:x2]\n        enlarged = cv2.resize(crop, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)\n        for box, text, confidence in reader.readtext(enlarged):\n            xs, ys = zip(*box, strict=True)\n            output.append(\n                {\n                    "kind": "ocr",\n                    "label": f"retry.{kind}",\n                    "text": text,\n                    "confidence": float(confidence),\n                    "x1": x1 + min(xs) / 3,\n                    "y1": y1 + min(ys) / 3,\n                    "x2": x1 + max(xs) / 3,\n                    "y2": y1 + max(ys) / 3,\n                }\n            )\n    return output\n',
    "pipeline.py": '"""YouTube acquisition, sampled OCR/detection proposals, resumable CSV exports."""\n\nimport csv\nimport hashlib\nimport json\nimport math\nimport re\nimport sqlite3\nfrom pathlib import Path\nfrom urllib.parse import parse_qs, urlparse\n\nVERSION = "video_v5"\nFIELDS = [\n    "observation_id",\n    "job_id",\n    "video_id",\n    "source_url",\n    "source_sha256",\n    "model_sha256",\n    "frame_index",\n    "timestamp_ms",\n    "kind",\n    "label",\n    "text",\n    "confidence",\n    "x1",\n    "y1",\n    "x2",\n    "y2",\n    "review_status",\n    "player_id",\n    "team_id",\n    "map_id",\n    "character_agent_id",\n    "pipeline_version",\n    "segment_type",\n    "processing_action",\n    "frame_width",\n    "frame_height",\n]\n\n\ndef channel_videos(\n    url, limit=5, section="videos", cookie_file=None, js_runtime=None, export_root=None\n):\n    """List a bounded channel page without downloading media."""\n    from yt_dlp import YoutubeDL\n    from yt_dlp.utils import DownloadError\n\n    from .acquisition import download_error_message, youtube_options\n\n    parsed = urlparse(url)\n    if (\n        parsed.scheme != "https"\n        or parsed.hostname not in {"youtube.com", "www.youtube.com"}\n        or not re.fullmatch(r"/@[A-Za-z0-9_.-]+/?", parsed.path)\n    ):\n        raise ValueError("Expected an HTTPS YouTube @channel URL")\n    if not 1 <= limit <= 50 or section not in {"videos", "streams"}:\n        raise ValueError("Limit must be 1–50 and section videos or streams")\n    target = f"https://www.youtube.com{parsed.path.rstrip(\'/\')}/{section}"\n    options = {\n        "extract_flat": True,\n        "playlistend": limit,\n        "skip_download": True,\n        **youtube_options(cookie_file, js_runtime, export_root),\n    }\n    try:\n        with YoutubeDL(options) as ydl:\n            listing = ydl.extract_info(target, download=False)\n    except DownloadError as error:\n        raise RuntimeError(download_error_message(error)) from None\n    return [\n        {\n            "video_id": item["id"],\n            "title": item.get("title", ""),\n            "url": f"https://www.youtube.com/watch?v={item[\'id\']}",\n        }\n        for item in listing.get("entries", [])\n        if item and re.fullmatch(r"[A-Za-z0-9_-]{11}", item.get("id", ""))\n    ][:limit]\n\n\ndef video_id(url):\n    parsed = urlparse(url)\n    host = (parsed.hostname or "").lower()\n    if parsed.scheme != "https" or host not in {"youtube.com", "www.youtube.com", "youtu.be"}:\n        raise ValueError("Use an HTTPS YouTube video link")\n    value = (\n        parsed.path.strip("/") if host == "youtu.be" else parse_qs(parsed.query).get("v", [""])[0]\n    )\n    if not re.fullmatch(r"[A-Za-z0-9_-]{11}", value):\n        raise ValueError("Expected a single watch?v= or youtu.be video link")\n    return value\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef download(url, root, cookie_file=None, js_runtime=None):\n    """Download a public video into the notebook runtime; do not bypass access restrictions."""\n    from yt_dlp import YoutubeDL\n    from yt_dlp.utils import DownloadError\n\n    from .acquisition import cached_video, download_error_message, youtube_options\n\n    key = video_id(url)\n    folder = Path(root) / "sources" / key\n    folder.mkdir(parents=True, exist_ok=True)\n    existing = cached_video(folder, key)\n    if existing:\n        print("Reusing completed video from cloud cache; no YouTube request needed.")\n        return existing\n    options = {\n        "noplaylist": True,\n        "format": "bv*[height<=1080]+ba/b[height<=1080]",\n        "merge_output_format": "mp4",\n        "outtmpl": str(folder / "video.%(ext)s"),\n        "writeinfojson": True,\n        "continuedl": True,\n        **youtube_options(cookie_file, js_runtime, root),\n    }\n    try:\n        with YoutubeDL(options) as downloader:\n            info = downloader.extract_info(f"https://www.youtube.com/watch?v={key}", download=True)\n            requested = info.get("requested_downloads") or []\n            candidates = [Path(info.get("filepath") or downloader.prepare_filename(info))]\n            candidates += [Path(item["filepath"]) for item in requested if item.get("filepath")]\n            candidates = [\n                folder / "video.mp4",\n                *candidates,\n                folder / "video.webm",\n                folder / "video.mkv",\n            ]\n            for candidate in candidates:\n                if candidate.is_file() and not re.search(r"\\.f\\d+\\.", candidate.name):\n                    return candidate\n    except DownloadError as error:\n        raise RuntimeError(download_error_message(error)) from None\n    raise RuntimeError("Download completed without a readable merged video")\n\n\ndef write_csv(path, rows):\n    target = Path(path)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = target.with_suffix(target.suffix + ".tmp")\n    with temporary.open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=FIELDS, extrasaction="raise")\n        writer.writeheader()\n        writer.writerows(rows)\n    temporary.replace(target)\n\n\nclass Job:\n    """One SQLite file per job. Commit each sampled frame, including empty frames."""\n\n    def __init__(self, root, identity):\n        self.identity = identity\n        self.id = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:24]\n        self.folder = Path(root) / "jobs" / self.id\n        self.folder.mkdir(parents=True, exist_ok=True)\n        self.db = sqlite3.connect(self.folder / "state.sqlite")\n        self.db.execute(\n            "CREATE TABLE IF NOT EXISTS frames (number INTEGER PRIMARY KEY, rows TEXT NOT NULL)"\n        )\n        (self.folder / "manifest.json").write_text(json.dumps(identity, indent=2), encoding="utf-8")\n\n    def done(self, number):\n        return (\n            self.db.execute("SELECT 1 FROM frames WHERE number=?", (number,)).fetchone() is not None\n        )\n\n    def restore_gate(self, number, gate):\n        encoded = self.db.execute("SELECT rows FROM frames WHERE number=?", (number,)).fetchone()[0]\n        for row in json.loads(encoded):\n            if row["kind"] == "segment":\n                gate.restore(json.loads(row["text"])["gate_state"])\n\n    def records(self, number, timestamp_ms, proposals):\n        rows = []\n        for i, proposal in enumerate(proposals):\n            row = dict.fromkeys(FIELDS, "")\n            row.update(proposal)\n            row.update(\n                observation_id=f"{self.id}:{number}:{i}",\n                job_id=self.id,\n                video_id=self.identity["video_id"],\n                source_url=self.identity["source_url"],\n                source_sha256=self.identity["source_sha256"],\n                model_sha256=self.identity["model_sha256"],\n                frame_index=number,\n                timestamp_ms=timestamp_ms,\n                review_status="proposed",\n                pipeline_version=VERSION,\n            )\n            rows.append(row)\n        return rows\n\n    def commit(self, number, timestamp_ms, proposals):\n        rows = self.records(number, timestamp_ms, proposals)\n        with self.db:\n            self.db.execute("INSERT INTO frames VALUES (?, ?)", (number, json.dumps(rows)))\n\n    def export(self):\n        def rows(segment=False):\n            for (encoded,) in self.db.execute("SELECT rows FROM frames ORDER BY number"):\n                yield from (\n                    row for row in json.loads(encoded) if (row["kind"] == "segment") == segment\n                )\n\n        path = self.folder / "observations.csv"\n        write_csv(path, rows())\n        write_csv(self.folder / "segments.csv", rows(segment=True))\n        from .segments import export_review_queue\n\n        export_review_queue(self.folder / "segments.csv")\n        from .roster import export_rosters\n\n        export_rosters(path, catalog=self.identity.get("roster_catalog"))\n        if getattr(self, "portrait_matcher", None) is not None:\n            from .portraits import classify_rosters\n\n            classify_rosters(path.with_name("roster_observations.csv"), self.portrait_matcher)\n        return path\n\n    def close(self):\n        self.db.close()\n\n\ndef process(\n    url,\n    root,\n    weights=None,\n    interval_seconds=2.0,\n    device="cpu",\n    end_seconds=None,\n    catalog=None,\n    segment_filter=True,\n    confirm_samples=2,\n    save_portrait_crops=True,\n    roster_catalog=None,\n    crop_ocr_retry=True,\n    portrait_reference_csv=None,\n    portrait_classifier_weights=None,\n    portrait_allowed_classes=None,\n    allow_proposed_portrait_references=False,\n    review_sample_limit=24,\n    cookie_file=None,\n    js_runtime=None,\n):\n    """OCR bootstrap works without weights; optional YOLO adds custom detector proposals."""\n    import cv2\n    import easyocr\n\n    from .entities import resolve, validate_catalog\n    from .ocr_retry import RETRY_VERSION, retry\n    from .roster import PROFILE_VERSION, SOURCE_CATALOG, parse_frame\n    from .segments import RULE_VERSION, SegmentGate, classify_probe\n\n    catalog = validate_catalog(catalog or [])\n    gate = SegmentGate(confirm_samples)\n    if not isinstance(review_sample_limit, int) or review_sample_limit < 0:\n        raise ValueError("review_sample_limit must be a nonnegative integer")\n    portrait_matcher = None\n    if portrait_reference_csv and portrait_classifier_weights:\n        raise ValueError("Choose reference matching or learned classification, not both")\n    if portrait_classifier_weights:\n        from .portraits import LearnedPortraitClassifier\n\n        if not save_portrait_crops:\n            raise ValueError("Portrait classification requires saved portrait crops")\n        portrait_matcher = LearnedPortraitClassifier(\n            portrait_classifier_weights, device=device, allowed_agent_ids=portrait_allowed_classes\n        )\n    if portrait_reference_csv:\n        from .portraits import PortraitMatcher\n\n        if not save_portrait_crops:\n            raise ValueError("Portrait classification requires saved portrait crops")\n        portrait_matcher = PortraitMatcher(\n            portrait_reference_csv, allow_proposed=allow_proposed_portrait_references\n        )\n\n    if not math.isfinite(interval_seconds) or interval_seconds <= 0:\n        raise ValueError("Sampling interval must be finite and positive")\n    if end_seconds is not None and (not math.isfinite(end_seconds) or end_seconds <= 0):\n        raise ValueError("End time must be finite and positive")\n    model = None\n    model_hash = "none-ocr-bootstrap"\n    if weights:\n        from ultralytics import YOLO\n\n        model_hash = sha256(weights)\n        model = YOLO(str(weights))\n        if model.task != "detect":\n            raise ValueError("Expected object detection weights")\n    source = (\n        download(url, root, cookie_file=cookie_file, js_runtime=js_runtime)\n        if cookie_file or js_runtime\n        else download(url, root)\n    )\n    key = video_id(url)\n    roster_catalog = roster_catalog or SOURCE_CATALOG.get(key)\n    import importlib.metadata\n\n    versions = {\n        name: importlib.metadata.version(name) for name in ["easyocr", "opencv-python-headless"]\n    }\n    if model:\n        versions["ultralytics"] = importlib.metadata.version("ultralytics")\n    job = Job(\n        root,\n        {\n            "pipeline_version": VERSION,\n            "video_id": key,\n            "source_url": f"https://www.youtube.com/watch?v={key}",\n            "source_sha256": sha256(source),\n            "model_sha256": model_hash,\n            "interval_seconds": interval_seconds,\n            "end_seconds": end_seconds,\n            "device": device,\n            "packages": versions,\n            "source_rights_status": "unconfirmed",\n            "is_synthetic": False,\n            "catalog": catalog,\n            "segment_filter": segment_filter,\n            "segment_rule_version": RULE_VERSION,\n            "confirm_samples": confirm_samples,\n            "roster_profile_version": PROFILE_VERSION,\n            "save_portrait_crops": save_portrait_crops,\n            "roster_catalog": roster_catalog,\n            "crop_ocr_retry": crop_ocr_retry,\n            "retry_version": RETRY_VERSION,\n            "portrait_matcher": portrait_matcher.identity if portrait_matcher else None,\n            "review_sample_limit": review_sample_limit,\n        },\n    )\n    job.portrait_matcher = portrait_matcher\n    review_folder = job.folder / "segment_review"\n    review_count = len(list(review_folder.glob("*.jpg")))\n    cap = cv2.VideoCapture(str(source))\n    try:\n        fps = cap.get(cv2.CAP_PROP_FPS)\n        if not cap.isOpened() or not math.isfinite(fps) or fps <= 0:\n            raise ValueError("Cannot decode video or determine frame rate")\n        step = max(1, round(fps * interval_seconds))\n        ocr_device = False if device == "cpu" else f"cuda:{device}"\n        reader = easyocr.Reader(["en"], gpu=ocr_device)\n        number = 0\n        processed = 0\n        while end_seconds is None or number / fps < end_seconds:\n            if not cap.grab():\n                break\n            if number % step == 0 and job.done(number):\n                job.restore_gate(number, gate)\n            elif number % step == 0:\n                ok, frame = cap.retrieve()\n                if not ok:\n                    raise RuntimeError(f"Frame decode failed at {number}")\n                proposals = []\n                segment_type, action, reason = "unfiltered", "extract", "Filter disabled"\n                probe = []\n                probe_items = []\n                if segment_filter:\n                    height = frame.shape[0]\n                    probe = reader.readtext(frame[: max(1, round(height * 0.24)), :])\n                    for box, text, confidence in probe:\n                        xs, ys = zip(*box, strict=True)\n                        probe_items.append(\n                            {\n                                "text": text,\n                                "confidence": float(confidence),\n                                "x1": float(min(xs)),\n                                "y1": float(min(ys)),\n                                "x2": float(max(xs)),\n                                "y2": float(max(ys)),\n                            }\n                        )\n                    candidate, reason, strong = classify_probe(probe_items, frame.shape[1], height)\n                    segment_type, action = gate.update(candidate, strong=strong)\n                proposals.append(\n                    {\n                        "kind": "segment",\n                        "label": segment_type,\n                        "segment_type": segment_type,\n                        "processing_action": action,\n                        "text": json.dumps(\n                            {\n                                "reason": reason,\n                                "gate_state": gate.state,\n                                "probe_text": [text for _, text, _ in probe],\n                                "probe_items": probe_items,\n                            }\n                        ),\n                    }\n                )\n                readings = reader.readtext(frame) if action == "extract" else []\n                for box, text, confidence in readings:\n                    xs, ys = zip(*box, strict=True)\n                    proposals.append(\n                        {\n                            "kind": "ocr",\n                            "label": "text_region",\n                            "text": text,\n                            "confidence": float(confidence),\n                            "x1": float(min(xs)),\n                            "y1": float(min(ys)),\n                            "x2": float(max(xs)),\n                            "y2": float(max(ys)),\n                        }\n                    )\n                if model and action == "extract":\n                    result = model.predict(frame, device=device, verbose=False, conf=0.25)[0]\n                    for box in result.boxes:\n                        x1, y1, x2, y2 = box.xyxy[0].tolist()\n                        proposals.append(\n                            {\n                                "kind": "detection",\n                                "label": result.names[int(box.cls.item())],\n                                "confidence": float(box.conf.item()),\n                                "x1": x1,\n                                "y1": y1,\n                                "x2": x2,\n                                "y2": y2,\n                            }\n                        )\n                if crop_ocr_retry and action == "extract" and roster_catalog is not None:\n                    shape = frame.shape\n                    records = job.records(number, round(number / fps * 1000), proposals)\n                    proposals.extend(retry(frame, reader, records, catalog=roster_catalog))\n                if number // step % 5 == 0 and review_count < review_sample_limit:\n                    review_folder.mkdir(exist_ok=True)\n                    review_image = review_folder / f"{number:09d}.jpg"\n                    if not cv2.imwrite(str(review_image), frame):\n                        raise RuntimeError("Could not save segment review frame")\n                    review_count += 1\n                    details = json.loads(proposals[0]["text"])\n                    details["review_image"] = f"segment_review/{review_image.name}"\n                    proposals[0]["text"] = json.dumps(details)\n                proposals.extend(resolve(proposals, catalog))\n                for proposal in proposals:\n                    proposal.update(segment_type=segment_type, processing_action=action)\n                    shape = getattr(frame, "shape", (1080, 1920))\n                    proposal.update(frame_width=shape[1], frame_height=shape[0])\n                if save_portrait_crops and action == "extract":\n                    # Save crops before committing the frame, so failed writes can retry.\n                    records = job.records(number, round(number / fps * 1000), proposals)\n                    for roster in parse_frame(\n                        records, width=shape[1], height=shape[0], catalog=roster_catalog\n                    ):\n                        crop = frame[\n                            roster["portrait_y1"] : roster["portrait_y2"],\n                            roster["portrait_x1"] : roster["portrait_x2"],\n                        ]\n                        folder = job.folder / "portrait_proposals"\n                        folder.mkdir(exist_ok=True)\n                        if not cv2.imwrite(\n                            str(folder / f"{roster[\'roster_observation_id\']}.jpg"), crop\n                        ):\n                            raise RuntimeError("Could not save portrait proposal")\n                job.commit(number, round(number / fps * 1000), proposals)\n                processed += 1\n                if processed % 25 == 0:\n                    job.export()\n                    print(\n                        f"Saved {processed} additional sampled frames; source time {number / fps:.1f}s"\n                    )\n            number += 1\n        if number == 0:\n            raise RuntimeError("Video decoded no frames")\n        return job.export()\n    finally:\n        cap.release()\n        job.export()\n        job.close()\n\n\ndef merge(inputs, output):\n    """Deduplicate identical jobs; retain different model versions and reject collisions."""\n    rows = {}\n    for path in inputs:\n        with Path(path).open(newline="", encoding="utf-8") as stream:\n            reader = csv.DictReader(stream)\n            if reader.fieldnames != FIELDS:\n                raise ValueError(f"Incompatible CSV schema: {path}")\n            for row in reader:\n                key = row["observation_id"]\n                if not key:\n                    raise ValueError("Missing observation ID")\n                if key in rows and rows[key] != row:\n                    raise ValueError(f"Conflicting observation: {key}")\n                rows[key] = row\n    write_csv(output, (rows[key] for key in sorted(rows)))\n    return Path(output)\n',
    "portraits.py": '"""Small reference-matching baseline with explicit abstention, not calibrated accuracy."""\n\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nREFERENCE_VERSION = "portrait_reference_v1"\n\n\ndef choose(scores, min_score=0.80, min_margin=0.08):\n    """Scores are feature cosine similarities, not probabilities."""\n    if not 0 <= min_score <= 1 or not 0 <= min_margin <= 2:\n        raise ValueError("Invalid reference thresholds")\n    best = {}\n    for label, score in scores:\n        if not label or not math.isfinite(score):\n            continue\n        best[label] = max(best.get(label, -1), score)\n    ranked = sorted(best.items(), key=lambda item: (-item[1], item[0]))\n    if len(ranked) < 2:\n        return {\n            "character_agent_id": "",\n            "score": None,\n            "margin": None,\n            "status": "insufficient_reference_classes",\n        }\n    (label, score), (_, runner_up) = ranked[:2]\n    margin = score - runner_up\n    accepted = score >= min_score and margin >= min_margin\n    return {\n        "character_agent_id": label if accepted else "",\n        "score": score,\n        "margin": margin,\n        "status": "proposed" if accepted else "unknown",\n    }\n\n\ndef feature(image):\n    import cv2\n    import numpy as np\n\n    # Remove lower HUD border, preserve aspect ratio, and normalize contrast.\n    gray = cv2.cvtColor(image[: max(1, round(image.shape[0] * 0.85))], cv2.COLOR_BGR2GRAY)\n    height, width = gray.shape\n    ratio = min(32 / width, 32 / height)\n    gray = cv2.resize(gray, (max(1, round(width * ratio)), max(1, round(height * ratio))))\n    canvas = np.zeros((32, 32), dtype=np.float32)\n    y, x = (32 - gray.shape[0]) // 2, (32 - gray.shape[1]) // 2\n    canvas[y : y + gray.shape[0], x : x + gray.shape[1]] = gray\n    vector = canvas.ravel()\n    vector -= vector.mean()\n    norm = float(np.linalg.norm(vector))\n    return vector / norm if norm > 1e-6 else None\n\n\nclass PortraitMatcher:\n    def __init__(self, reference_csv, allow_proposed=False, min_score=0.80, min_margin=0.08):\n        import cv2\n\n        self.min_score, self.min_margin = min_score, min_margin\n        choose([], min_score, min_margin)\n        path = Path(reference_csv).resolve()\n        root = path.parent\n        self.references = []\n        digest = hashlib.sha256(path.read_bytes())\n        with path.open(newline="", encoding="utf-8") as stream:\n            for row in csv.DictReader(stream):\n                reviewed = (\n                    row.get("label_status") == "reviewed"\n                    and row.get("independent_review_status") == "reviewed"\n                    and bool(row.get("reviewer_id"))\n                )\n                if not reviewed and not (allow_proposed and row.get("label_status") == "proposed"):\n                    continue\n                if not row.get("character_agent_id"):\n                    continue\n                image_path = (root / row["image"]).resolve()\n                if not image_path.is_relative_to(root):\n                    raise ValueError("Reference image is outside its dataset folder")\n                image = cv2.imread(str(image_path))\n                if image is None:\n                    raise ValueError(f"Unreadable reference image: {image_path.name}")\n                vector = feature(image)\n                if vector is not None:\n                    self.references.append((row["character_agent_id"], vector))\n                    digest.update(image_path.read_bytes())\n        if len({label for label, _ in self.references}) < 2:\n            raise ValueError(\n                "Need at least two eligible reference classes; independent review is required by default"\n            )\n        self.identity = {\n            "version": REFERENCE_VERSION,\n            "sha256": digest.hexdigest(),\n            "allow_proposed": allow_proposed,\n            "min_score": min_score,\n            "min_margin": min_margin,\n            "method": "grayscale_cosine_reference",\n        }\n\n    def predict(self, image):\n        vector = feature(image)\n        scores = (\n            []\n            if vector is None\n            else [(label, float(vector @ ref)) for label, ref in self.references]\n        )\n        return choose(scores, self.min_score, self.min_margin)\n\n\nclass LearnedPortraitClassifier:\n    """Adapter for separately trained YOLO classification weights, with abstention."""\n\n    def __init__(\n        self, weights, device="0", min_score=0.80, min_margin=0.15, allowed_agent_ids=None\n    ):\n        from ultralytics import YOLO\n\n        path = Path(weights)\n        self.model = YOLO(str(path))\n        if self.model.task != "classify":\n            raise ValueError("Portrait weights must be an image classifier, not a HUD detector")\n        allowed = set(allowed_agent_ids or ["chamber", "viper", "fade", "omen", "neon", "yoru"])\n        allowed |= {"unknown", "other", "occluded"}\n        if any(label not in allowed for label in self.model.names.values()):\n            raise ValueError(\n                "Classifier contains unapproved character classes; do not use generic ImageNet weights for extraction"\n            )\n        self.device = device\n        self.min_score, self.min_margin = min_score, min_margin\n        choose([], min_score, min_margin)\n        self.identity = {\n            "version": "portrait_classifier_v1",\n            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),\n            "method": "yolo_classification",\n            "min_score": min_score,\n            "min_margin": min_margin,\n            "allowed_classes": sorted(allowed),\n        }\n\n    def predict(self, image):\n        result = self.model.predict(image, device=self.device, verbose=False)[0]\n        values = result.probs.data.tolist()\n        scores = [(self.model.names[i], float(value)) for i, value in enumerate(values)]\n        output = choose(scores, self.min_score, self.min_margin)\n        if output["character_agent_id"].casefold() in {"unknown", "other", "occluded"}:\n            output.update(character_agent_id="", status="unknown")\n        return output\n\n\ndef classify_rosters(roster_csv, matcher):\n    import cv2\n\n    path = Path(roster_csv)\n    with path.open(newline="", encoding="utf-8") as stream:\n        reader = csv.DictReader(stream)\n        fields = list(reader.fieldnames or [])\n        rows = list(reader)\n    # Keep roster output schema separate from optional portrait enrichment.\n    fields = list(rows[0]) if rows else fields\n    fields += ["portrait_status", "portrait_score", "portrait_margin", "portrait_model_version"]\n    for row in rows:\n        crop = path.parent / "portrait_proposals" / (row["roster_observation_id"] + ".jpg")\n        image = cv2.imread(str(crop)) if crop.is_file() else None\n        prediction = (\n            matcher.predict(image)\n            if image is not None\n            else {"character_agent_id": "", "status": "missing_crop", "score": None, "margin": None}\n        )\n        row.update(\n            character_agent_id=prediction["character_agent_id"],\n            portrait_status=prediction["status"],\n            portrait_score=prediction["score"],\n            portrait_margin=prediction["margin"],\n            portrait_model_version=matcher.identity["version"],\n        )\n    target = path.with_name("roster_with_agents.csv")\n    if not fields:\n        return None\n    with target.with_suffix(".csv.tmp").open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(rows)\n    target.with_suffix(".csv.tmp").replace(target)\n    (path.parent / "portrait_model.json").write_text(\n        json.dumps(matcher.identity, indent=2), encoding="utf-8"\n    )\n    return target\n',
    "roster.py": '"""Spatial VCT roster proposals. No membership or pick is inferred from world labels."""\n\nimport csv\nimport hashlib\nfrom collections import defaultdict\nfrom pathlib import Path\n\nfrom .entities import normalize\n\nPROFILE_VERSION = "vct_1080_v3"\nNAMES = [\n    "Cryo",\n    "bang",\n    "Vora",\n    "Asuna",\n    "Timotino",\n    "Chronicle",\n    "Derke",\n    "Jamppi",\n    "Sayonara",\n    "PROFEK",\n]\n# This seed is scoped to the inspected source; it is not a global player directory.\nSOURCE_CATALOG = {\n    "omjZXmc7_4A": {\n        "players": {\n            normalize(name): f"source_handle:omjZXmc7_4A:{normalize(name)}" for name in NAMES\n        },\n        "teams": {"100t": "100t", "i0ot": "100t", "10ot": "100t", "iqot": "100t", "vit": "vit"},\n        "maps": {"split": "split", "summit": "summit", "lotus": "lotus"},\n    }\n}\nROSTER_FIELDS = [\n    "roster_observation_id",\n    "source_url",\n    "video_id",\n    "timestamp_ms",\n    "frame_index",\n    "map_id",\n    "team_id",\n    "opposing_team_id",\n    "player_id",\n    "display_name",\n    "character_agent_id",\n    "side",\n    "slot",\n    "layout",\n    "name_confidence",\n    "team_confidence",\n    "map_confidence",\n    "review_status",\n    "evidence_observation_id",\n    "portrait_x1",\n    "portrait_y1",\n    "portrait_x2",\n    "portrait_y2",\n    "profile_version",\n]\n\n\ndef inside(row, region, width, height):\n    x = (float(row["x1"]) + float(row["x2"])) / 2 * 1920 / width\n    y = (float(row["y1"]) + float(row["y2"])) / 2 * 1080 / height\n    x1, y1, x2, y2 = region\n    return x1 <= x <= x2 and y1 <= y <= y2\n\n\ndef unique(candidates):\n    """Conflicting OCR identities in one region require review, not a best guess."""\n    if not candidates or len({value for value, _ in candidates}) != 1:\n        return "", ""\n    return candidates[0][0], max(confidence for _, confidence in candidates)\n\n\ndef parse_frame(rows, width=1920, height=1080, catalog=None):\n    if width <= 0 or height <= 0:\n        raise ValueError("Frame dimensions must be positive")\n    if not rows:\n        return []\n    source = rows[0]["video_id"]\n    catalog = catalog or SOURCE_CATALOG.get(source)\n    if catalog is None:\n        return []  # Unknown broadcast needs a reviewed catalog, not guessed identities.\n    ocr = [row for row in rows if row["kind"] == "ocr"]\n    teams = []\n    for region in [(710, 0, 795, 55), (1125, 0, 1200, 55)]:\n        candidates = [\n            (catalog["teams"][normalize(row["text"])], float(row["confidence"]))\n            for row in ocr\n            if float(row["confidence"]) >= 0.2\n            and normalize(row["text"]) in catalog["teams"]\n            and inside(row, region, width, height)\n        ]\n        teams.append(unique(candidates))\n    if teams[0][0] and teams[0][0] == teams[1][0]:\n        teams = [("", ""), ("", "")]  # Two scoreboard sides cannot establish one team.\n    maps = []\n    for row in ocr:\n        text = normalize(row["text"])\n        if text.startswith("current ") and inside(row, (0, 0, 220, 35), width, height):\n            name = text.removeprefix("current ")\n            if name in catalog["maps"] and float(row["confidence"]) >= 0.5:\n                maps.append((catalog["maps"][name], float(row["confidence"])))\n    map_id, map_confidence = unique(maps)\n    output = []\n    for layout in ["buy", "sidebar"]:\n        for side in range(2):\n            for slot in range(5):\n                if layout == "buy":\n                    y = 783 + slot * 56\n                    region = (645, y, 815, y + 53) if side == 0 else (1165, y, 1275, y + 53)\n                    portrait = (573, y, 641, y + 53) if side == 0 else (1276, y, 1348, y + 53)\n                else:\n                    y = 550 + slot * 106\n                    region = (77, y, 255, y + 45) if side == 0 else (1725, y, 1841, y + 45)\n                    portrait = (17, y, 76, y + 47) if side == 0 else (1841, y, 1905, y + 47)\n                candidates = [\n                    row\n                    for row in ocr\n                    if normalize(row["text"]) in catalog["players"]\n                    and float(row["confidence"]) >= 0.5\n                    and inside(row, region, width, height)\n                ]\n                player, confidence = unique(\n                    [\n                        (catalog["players"][normalize(row["text"])], float(row["confidence"]))\n                        for row in candidates\n                    ]\n                )\n                if not player:\n                    continue\n                evidence = max(candidates, key=lambda row: float(row["confidence"]))\n                identity = f"{evidence[\'observation_id\']}:{PROFILE_VERSION}:{layout}:{side}:{slot}"\n                row = {field: "" for field in ROSTER_FIELDS}\n                row.update(\n                    roster_observation_id=hashlib.sha256(identity.encode()).hexdigest()[:24],\n                    source_url=evidence["source_url"],\n                    video_id=source,\n                    timestamp_ms=evidence["timestamp_ms"],\n                    frame_index=evidence["frame_index"],\n                    map_id=map_id,\n                    map_confidence=map_confidence,\n                    team_id=teams[side][0],\n                    team_confidence=teams[side][1],\n                    opposing_team_id=teams[1 - side][0],\n                    player_id=player,\n                    display_name=evidence["text"],\n                    side=["left", "right"][side],\n                    slot=slot + 1,\n                    layout=layout,\n                    name_confidence=confidence,\n                    review_status="proposed",\n                    evidence_observation_id=evidence["observation_id"],\n                    profile_version=PROFILE_VERSION,\n                )\n                for field, value in zip(\n                    ["portrait_x1", "portrait_y1", "portrait_x2", "portrait_y2"],\n                    portrait,\n                    strict=True,\n                ):\n                    row[field] = round(\n                        value * (width / 1920 if field.endswith(("x1", "x2")) else height / 1080)\n                    )\n                output.append(row)\n    # Two possible layouts or slots for one identity is ambiguous.\n    counts = defaultdict(int)\n    for row in output:\n        counts[row["player_id"]] += 1\n    return [row for row in output if counts[row["player_id"]] == 1]\n\n\ndef export_rosters(observations, output=None, width=1920, height=1080, catalog=None):\n    groups = defaultdict(list)\n    with Path(observations).open(newline="", encoding="utf-8") as stream:\n        for row in csv.DictReader(stream):\n            if row.get("processing_action") == "extract":\n                groups[(row["video_id"], int(row["frame_index"]))].append(row)\n    result = []\n    for key in sorted(groups):\n        group = groups[key]\n        result.extend(\n            parse_frame(\n                group,\n                width=int(group[0].get("frame_width") or width),\n                height=int(group[0].get("frame_height") or height),\n                catalog=catalog,\n            )\n        )\n    target = Path(output) if output else Path(observations).with_name("roster_observations.csv")\n    temp = target.with_suffix(".csv.tmp")\n    with temp.open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=ROSTER_FIELDS)\n        writer.writeheader()\n        writer.writerows(result)\n    temp.replace(target)\n    return target\n',
    "segments.py": '"""Conservative broadcast rules; unknown layouts remain reviewable, never guessed."""\n\nimport csv\nimport json\nimport re\nfrom pathlib import Path\n\nRULE_VERSION = "hud_rules_v3"\n\n\ndef export_review_queue(segment_csv):\n    path = Path(segment_csv)\n    fields = [\n        "video_id",\n        "timestamp_ms",\n        "image",\n        "suggested_segment",\n        "processing_action",\n        "reason",\n        "reference_segment",\n        "review_status",\n        "reviewer_id",\n    ]\n    # Reviewers copy this generated file to segment_review_labels.csv before editing.\n    target = path.with_name("segment_review_suggestions.csv")\n    temporary = target.with_suffix(".csv.tmp")\n    with (\n        path.open(newline="", encoding="utf-8") as source,\n        temporary.open("w", newline="", encoding="utf-8") as dest,\n    ):\n        writer = csv.DictWriter(dest, fieldnames=fields)\n        writer.writeheader()\n        for row in csv.DictReader(source):\n            details = json.loads(row["text"])\n            if details.get("review_image"):\n                writer.writerow(\n                    {\n                        "video_id": row["video_id"],\n                        "timestamp_ms": row["timestamp_ms"],\n                        "image": details["review_image"],\n                        "suggested_segment": row["segment_type"],\n                        "processing_action": row["processing_action"],\n                        "reason": details["reason"],\n                        "reference_segment": "",\n                        "review_status": "pending",\n                        "reviewer_id": "",\n                    }\n                )\n    temporary.replace(target)\n    return target\n\n\ndef classify(texts):\n    """Use a small upper-HUD OCR probe, not a full-frame detector, on every sample."""\n    tokens = [text.strip().casefold() for text in texts]\n    combined = " ".join(tokens)\n    if re.search(r"\\breplay\\b", combined):\n        return "replay", "Replay marker visible"\n    if re.search(r"\\b(starting lineup|agent select|map veto|map pick|team roster)\\b", combined):\n        return "roster", "Roster or selection marker visible"\n    if re.search(\n        r"\\b(be right back|we.?ll be right back|starting soon|stream starting|"\n        r"technical pause|halftime|half time|match paused|commercial break|"\n        r"analyst desk|up next|stay tuned|intermission)\\b",\n        combined,\n    ):\n        return "filler", "Break, desk or pause marker visible"\n    timer = any(re.fullmatch(r"\\d{1,2}[:.]\\d{2}", text) for text in tokens)\n    # VCT\'s map-order strip and round label remain visible when score OCR fails.\n    round_label = bool(re.search(r"\\bround\\s*\\d{1,2}\\b", combined))\n    map_order = bool(re.search(r"\\bcurrent\\s*[:.]\\s*\\w+", combined)) and bool(\n        re.search(r"\\bnext\\s*[:.]\\s*\\w+", combined)\n    )\n    if round_label and map_order:\n        return "gameplay", "VCT map-order strip and round label visible"\n    scores = sum(bool(re.fullmatch(r"\\d{1,2}", text)) for text in tokens)\n    if timer and scores >= 2:\n        return "gameplay", "Upper HUD timer and two score candidates visible"\n    return "uncertain", "No reliable broadcast-state signal"\n\n\ndef classify_probe(items, width, height):\n    """Strong same-frame HUD groups can confirm a cut without carrying old state."""\n    candidate, reason = classify([item["text"] for item in items if item["confidence"] >= 0.3])\n    if candidate in {"filler", "replay", "roster"}:\n        return candidate, reason, False\n\n    def located(item, area):\n        if not all(key in item for key in ["x1", "y1", "x2", "y2"]):\n            return True  # Legacy upper-HUD transcripts: usable only for diagnostic replay.\n        x = (item["x1"] + item["x2"]) / 2 / width\n        y = (item["y1"] + item["y2"]) / 2 / height\n        x1, y1, x2, y2 = area\n        return x1 <= x <= x2 and y1 <= y <= y2\n\n    map_text = [\n        item["text"].strip().casefold()\n        for item in items\n        if item["confidence"] >= 0.5 and located(item, (0, 0, 0.34, 0.06))\n    ]\n    map_order = any(re.match(r"current\\s*[:.]\\s*\\w+", text) for text in map_text) and any(\n        re.match(r"next\\s*[:.]\\s*\\w+", text) for text in map_text\n    )\n    clock = False\n    round_label = False\n    for item in items:\n        text = item["text"].strip().casefold()\n        if not located(item, (0.40, 0, 0.60, 0.10)):\n            continue\n        match = re.fullmatch(r"\\d{1,2}[:.](\\d{2})", text)\n        clock |= bool(match and int(match[1]) < 60 and item["confidence"] >= 0.5)\n        round_label |= bool(re.fullmatch(r"round\\s*\\d{1,2}", text) and item["confidence"] >= 0.2)\n    if map_order and (clock or round_label):\n        return "gameplay", "Same-frame map strip and central clock/round HUD", True\n    return candidate, reason, False\n\n\nclass SegmentGate:\n    """Confirm transitions across samples; suppress extraction during ambiguity."""\n\n    def __init__(self, confirm_samples=2):\n        if (\n            not isinstance(confirm_samples, int)\n            or isinstance(confirm_samples, bool)\n            or confirm_samples < 1\n        ):\n            raise ValueError("confirm_samples must be a positive integer")\n        self.required = confirm_samples\n        self.state = {"stable": "uncertain", "candidate": "uncertain", "count": 0}\n\n    def update(self, candidate, strong=False):\n        if candidate not in {"gameplay", "roster", "replay", "filler", "uncertain"}:\n            raise ValueError("Unknown segment type")\n        count = self.state["count"] + 1 if candidate == self.state["candidate"] else 1\n        stable = candidate if count >= self.required else self.state["stable"]\n        if strong and candidate == "gameplay":\n            stable = candidate\n            count = max(count, self.required)\n        self.state = {"stable": stable, "candidate": candidate, "count": count}\n        # A previously stable match must not admit the first frame of an ad/replay.\n        if candidate in {"replay", "filler"}:\n            return candidate, "skip"\n        if candidate in {"gameplay", "roster"} and stable == candidate and count >= self.required:\n            return candidate, "extract"\n        return "uncertain", "review"\n\n    def restore(self, state):\n        self.state = dict(state)\n',
    "__init__.py": '"""Portable notebook processing; heavy GPU dependencies are loaded only when used."""\n',
}
helpers = {
    "prepare_portrait_dataset.py": '"""Build a classifier dataset from independently reviewed, source-separated labels."""\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport re\nimport shutil\nfrom pathlib import Path\n\n\ndef prepare(review_csv, output):\n    source = Path(review_csv).resolve()\n    root = source.parent\n    output = Path(output)\n    if output.exists():\n        raise ValueError("Output must be a new directory")\n    with source.open(newline="", encoding="utf-8") as stream:\n        rows = list(csv.DictReader(stream))\n    eligible = [\n        row\n        for row in rows\n        if row.get("label_status") == "reviewed"\n        and row.get("independent_review_status") == "reviewed"\n        and row.get("reviewer_id")\n    ]\n    if not eligible:\n        raise ValueError("No independently reviewed character labels")\n    groups = {}\n    planned = []\n    for row in eligible:\n        video, label, split = (\n            row.get("video_id"),\n            row.get("character_agent_id", ""),\n            row.get("split"),\n        )\n        if (\n            not video\n            or split not in {"train", "val", "test"}\n            or not re.fullmatch(r"[a-z][a-z0-9_]*", label)\n        ):\n            raise ValueError(\n                "Each reviewed row needs video_id, character label and explicit train/val/test split"\n            )\n        if video in groups and groups[video] != split:\n            raise ValueError("All images from one source video must remain in one split")\n        groups[video] = split\n        image = (root / row["image"]).resolve()\n        if not image.is_relative_to(root) or not image.is_file():\n            raise ValueError("Missing image or image outside dataset folder")\n        planned.append((image, split, label))\n    classes = {\n        split: {label for _, partition, label in planned if partition == split}\n        for split in ["train", "val"]\n    }\n    if len(classes["train"]) < 2 or classes["train"] != classes["val"]:\n        raise ValueError("Train and validation must cover the same two or more character classes")\n    seen_hashes = {}\n    for image, split, _ in planned:\n        digest = hashlib.sha256(image.read_bytes()).hexdigest()\n        if digest in seen_hashes and seen_hashes[digest] != split:\n            raise ValueError("Identical image bytes cross dataset splits")\n        seen_hashes[digest] = split\n    output.mkdir(parents=True)\n    for image, split, label in planned:\n        target = output / split / label\n        target.mkdir(parents=True, exist_ok=True)\n        digest = hashlib.sha256(str(image).encode()).hexdigest()[:12]\n        shutil.copyfile(image, target / (digest + image.suffix))\n    (output / "dataset_manifest.json").write_text(\n        json.dumps(\n            {\n                "schema_version": "portrait_dataset_v1",\n                "source_csv_sha256": hashlib.sha256(source.read_bytes()).hexdigest(),\n                "source_splits": groups,\n                "classes": sorted(classes["train"]),\n                "images": len(planned),\n                "note": "Source grouping does not independently guarantee match separation; check mirrored/edited uploads.",\n            },\n            indent=2,\n        ),\n        encoding="utf-8",\n    )\n    return output\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("review_csv", type=Path)\n    parser.add_argument("output", type=Path)\n    args = parser.parse_args()\n    print(prepare(args.review_csv, args.output))\n\n\nif __name__ == "__main__":\n    main()\n',
    "evaluate_segment_review.py": '"""Evaluate only explicitly reviewed segment samples; preserve abstention counts."""\n\nimport argparse\nimport csv\nimport json\nfrom collections import Counter\nfrom pathlib import Path\n\n\ndef evaluate(rows):\n    labels = {"gameplay", "roster", "filler", "replay", "uncertain"}\n    reviewed = [\n        row for row in rows if row.get("review_status") == "reviewed" and row.get("reviewer_id")\n    ]\n    if not reviewed:\n        raise ValueError("No reviewed segment references")\n    counts = Counter()\n    misses = 0\n    admitted_nonmatch = 0\n    for row in reviewed:\n        expected, actual = row["reference_segment"], row["suggested_segment"]\n        if expected not in labels or actual not in labels:\n            raise ValueError("Invalid segment class")\n        counts[(expected, actual)] += 1\n        misses += expected in {"gameplay", "roster"} and row["processing_action"] != "extract"\n        admitted_nonmatch += (\n            expected in {"filler", "replay"} and row["processing_action"] == "extract"\n        )\n    return {\n        "reviewed_samples": len(reviewed),\n        "matching_labels": sum(\n            a == b\n            for a, b in [(row["reference_segment"], row["suggested_segment"]) for row in reviewed]\n        ),\n        "missed_match_samples": misses,\n        "admitted_filler_or_replay_samples": admitted_nonmatch,\n        "confusion": [\n            {"reference": a, "predicted": b, "count": count}\n            for (a, b), count in sorted(counts.items())\n        ],\n        "scope": "Reviewed sampled frames only; no full-video accuracy claim",\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("review_csv", type=Path)\n    args = parser.parse_args()\n    with args.review_csv.open(newline="", encoding="utf-8") as stream:\n        result = evaluate(list(csv.DictReader(stream)))\n    output = args.review_csv.with_suffix(".evaluation.json")\n    output.write_text(json.dumps(result, indent=2), encoding="utf-8")\n    print(json.dumps(result, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n',
}
package = Path("/tmp/valomind_worker/valomind_video")
package.mkdir(parents=True, exist_ok=True)
for name, source in modules.items():
    (package / name).write_text(source, encoding="utf-8")
for name, source in helpers.items():
    (package.parent / name).write_text(source, encoding="utf-8")
sys.path.insert(0, str(package.parent))
for name in list(sys.modules):
    if (
        name == "valomind_video"
        or name.startswith("valomind_video.")
        or name in {"prepare_portrait_dataset", "evaluate_segment_review"}
    ):
        del sys.modules[name]
from valomind_video.pipeline import channel_videos, download, process, merge
from valomind_video.roster import export_rosters
from valomind_video.acquisition import ensure_runtime, restore_source_archive
from valomind_video.pipeline import video_id
from prepare_portrait_dataset import prepare as prepare_classifier_dataset
from evaluate_segment_review import evaluate as evaluate_segments

JS_RUNTIME = ensure_runtime()
print("yt-dlp JavaScript runtime:", ", ".join(JS_RUNTIME))

In [ ]:
# Colab: optionally mount Drive first and set WORK_ROOT under /content/drive/MyDrive/.
# Kaggle: /kaggle/working/valomind is suitable for saved notebook outputs.
WORK_ROOT = Path("/kaggle/working/valomind" if Path("/kaggle").exists() else "/content/valomind")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_URL = "https://youtu.be/omjZXmc7_4A"
CHANNEL_SECTION = "streams"  # 'videos' for uploads/highlights
LIST_LIMIT = 5
COOKIE_FILE = (
    None  # Optional own-session Netscape cookies file outside WORK_ROOT. Never paste it here.
)
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU assigned. In Colab select Runtime > Change runtime type > T4 GPU, then rerun."
    )
torch.cuda.set_device(0)
DEVICE = "0"  # Exactly one GPU for YOLO inference/training; no multi-GPU list.
GPU_NAME = torch.cuda.get_device_name(0)
print("Using CUDA GPU 0:", GPU_NAME)
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
if "T4" not in GPU_NAME.upper():
    print("This runtime assigned a different GPU; only GPU 0 will be used.")
# Verify a CUDA operation before downloading the video.
probe = torch.ones(1, device="cuda:0") * 2
torch.cuda.synchronize(0)
assert probe.item() == 2
del probe
frozen = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(WORK_ROOT / "environment.txt").write_text(frozen)

if "/@" in SOURCE_URL:
    candidates = channel_videos(
        SOURCE_URL,
        limit=LIST_LIMIT,
        section=CHANNEL_SECTION,
        cookie_file=COOKIE_FILE,
        js_runtime=JS_RUNTIME,
        export_root=WORK_ROOT,
    )
    for index, item in enumerate(candidates):
        print(index, item["title"], item["url"])
    SELECTED_URLS = []  # Paste selected candidate URLs here, then run the extraction cell.
else:
    SELECTED_URLS = [SOURCE_URL]

## Recover this existing video if YouTube blocks Colab
The FIRST valomind_results.zip contains the full previously downloaded video.
Later numbered exports contain metadata only. Put that first ZIP in your private
Google Drive, set SOURCE_ARCHIVE below, and mount Drive when requested. Restoring
copies only the selected source video/metadata and verifies its archived checksum
and duration. Processing remains on the cloud GPU; no new YouTube request is used.
This recovery applies to the existing video, not new blocked YouTube links.


In [ ]:
SOURCE_ARCHIVE = None  # e.g. '/content/drive/MyDrive/valomind_results.zip'
if SOURCE_ARCHIVE:
    if str(SOURCE_ARCHIVE).startswith("/content/drive/"):
        from google.colab import drive

        drive.mount("/content/drive")
    if not SELECTED_URLS:
        raise ValueError("Select the source video URL before restoring its archive.")
    for url in SELECTED_URLS:
        print(restore_source_archive(SOURCE_ARCHIVE, WORK_ROOT, video_id(url)))

In [ ]:
# Configure this cell after inspecting the channel listing.
# SELECTED_URLS = ['https://www.youtube.com/watch?v=...']
WEIGHTS = None  # Path to custom best.pt. None runs OCR bootstrap only.
CATALOG = []  # e.g. {'kind':'player','id':'p1','aliases':['EXACT DISPLAY NAME']}
# Other kinds: team, map, character_agent. Use consistent IDs across notebooks.
INTERVAL_SECONDS = 2.0
END_SECONDS = 120  # Preview first; set None for the full video.
SEGMENT_FILTER = True  # Skip detected fillers/replays before full OCR and YOLO.
CONFIRM_SAMPLES = (
    2  # Weak HUD evidence needs consecutive samples; strong spatial HUD can admit one.
)
SAVE_PORTRAIT_CROPS = True  # Unlabeled roster portraits for review/classifier training.
ROSTER_CATALOG = None  # None uses the inspected video's scoped name/team aliases.
CROP_OCR_RETRY = True  # Retry unresolved team/name regions at 3x scale.
PORTRAIT_REFERENCE_CSV = None  # Reviewed portrait_review.csv from an extracted reference pack.
PORTRAIT_CLASSIFIER_WEIGHTS = (
    None  # Trained character classifier best.pt; different from HUD WEIGHTS.
)
PORTRAIT_ALLOWED_CLASSES = None  # Defaults to the six inspected characters plus unknown.
ALLOW_PROPOSED_PORTRAIT_REFERENCES = False  # Diagnostic use only; never verifies outputs.
REVIEW_SAMPLE_LIMIT = 24  # All segment states: JPEGs sampled every fifth probe.
if not SELECTED_URLS:
    raise ValueError("Select one or more video URLs from the channel listing first.")
outputs = []
for url in SELECTED_URLS:
    output = process(
        url,
        WORK_ROOT,
        weights=WEIGHTS,
        device=DEVICE,
        interval_seconds=INTERVAL_SECONDS,
        end_seconds=END_SECONDS,
        catalog=CATALOG,
        segment_filter=SEGMENT_FILTER,
        confirm_samples=CONFIRM_SAMPLES,
        save_portrait_crops=SAVE_PORTRAIT_CROPS,
        roster_catalog=ROSTER_CATALOG,
        crop_ocr_retry=CROP_OCR_RETRY,
        portrait_reference_csv=PORTRAIT_REFERENCE_CSV,
        portrait_classifier_weights=PORTRAIT_CLASSIFIER_WEIGHTS,
        portrait_allowed_classes=PORTRAIT_ALLOWED_CLASSES,
        allow_proposed_portrait_references=ALLOW_PROPOSED_PORTRAIT_REFERENCES,
        review_sample_limit=REVIEW_SAMPLE_LIMIT,
        cookie_file=COOKIE_FILE,
        js_runtime=JS_RUNTIME,
    )
    outputs.append(output)
    print("Saved", output)
    print("Segment decisions:", output.parent / "segments.csv")
    print("Player/team/map proposals:", output.parent / "roster_observations.csv")
    if PORTRAIT_REFERENCE_CSV or PORTRAIT_CLASSIFIER_WEIGHTS:
        print("Agent proposals:", output.parent / "roster_with_agents.csv")
    print("Segment review images:", output.parent / "segment_review")
    print("Segment labeling queue:", output.parent / "segment_review_suggestions.csv")
    import csv
    from collections import Counter

    with (output.parent / "segments.csv").open(newline="", encoding="utf-8") as stream:
        decisions = list(csv.DictReader(stream))
    counts = Counter(row["segment_type"] for row in decisions)
    print("Sampled-frame decisions:", dict(counts))
    print("These are sampled-frame counts, not exact segment durations.")
    if not any(row["processing_action"] == "extract" for row in decisions):
        print(
            "No gameplay/roster extraction admitted. Inspect probe text in segments.csv; "
            "the preview may be an intro, or this HUD layout may need tuning."
        )

## Broadcast filtering
Filtering is enabled by default. An upper-HUD OCR probe looks for timer/score,
roster/selection labels, replay markers and break/desk messages. Strong same-frame
map-strip plus central timer/round evidence admits gameplay immediately; weaker
signals require consecutive samples. Filler and replay markers suppress
extraction immediately; unclear frames are recorded for review, not guessed.
`segments.csv` records every sampled decision and probe text. Replays are skipped
from the observation CSV, not automatically deduplicated when no replay marker is
visible. Unfamiliar layouts or unmarked roster screens can be missed. Review the
segment CSV before full runs; this is a heuristic baseline, not a trained segment
classifier. Timing resolution follows INTERVAL_SECONDS, not exact video cuts.
If needed, set SEGMENT_FILTER=False for a diagnostic unfiltered extraction.
The entire video is still downloaded and decoded; filtering reduces full-frame
OCR/detector work, not download size. Pending review frames retain probe text only.
The default URL is the user-selected one-hour highlight compilation. Start with
END_SECONDS=120, inspect the summary/segment CSV, then set END_SECONDS=None for
the full source. Highlight cuts may omit roster screens and distort behavior rates.


## Optional: create labeling frames, then train
This does not auto-label ground truth. Label the sampled images and prepare a
YOLO dataset YAML with separate train/validation partitions. Use different matches
for meaningful validation. Do not feed raw OCR output back as verified labels.


In [ ]:
MAKE_LABELING_FRAMES = False
if MAKE_LABELING_FRAMES:
    import cv2

    url = SELECTED_URLS[0]
    video = download(url, WORK_ROOT, cookie_file=COOKIE_FILE, js_runtime=JS_RUNTIME)
    folder = WORK_ROOT / "labeling" / video.parent.name
    folder.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video))
    try:
        fps = cap.get(cv2.CAP_PROP_FPS)
        if not cap.isOpened() or fps <= 0:
            raise ValueError("Cannot decode source video")
        for second in range(0, 300, 5):
            cap.set(cv2.CAP_PROP_POS_MSEC, second * 1000)
            ok, frame = cap.read()
            if not ok:
                break
            if not cv2.imwrite(str(folder / f"{second * 1000:09d}.jpg"), frame):
                raise RuntimeError("Could not save labeling frame")
    finally:
        cap.release()
    print("Label these images:", folder)

In [ ]:
TRAIN = False
DATA_YAML = ""  # Prepared, reviewed YOLO labels required.
INITIAL_WEIGHTS = "yolo11n.pt"
if TRAIN:
    from ultralytics import YOLO

    if not Path(DATA_YAML).is_file():
        raise ValueError("Provide a labeled YOLO dataset YAML")
    model = YOLO(INITIAL_WEIGHTS)
    model.train(
        data=DATA_YAML,
        epochs=30,
        imgsz=960,
        batch=4,
        device=DEVICE,
        amp=True,
        seed=42,
        project=str(WORK_ROOT / "training"),
        name="hud",
        exist_ok=False,
    )
    print("Use the reported best.pt path as WEIGHTS for extraction.")

## Optional: train the separate character classifier
Use a reviewed dataset with train/character and val/character folders. Keep whole
videos/matches in one partition. scripts/prepare_portrait_dataset.py validates
source splits and excludes pending labels. The current 20-image assistant pack
is not independently reviewed and is not a generalization benchmark. The simple
reference matcher is diagnostic: its first cross-layout check abstained on all
10 query portraits. Do not lower its thresholds just to fill agent fields.


In [ ]:
TRAIN_PORTRAITS = False
PORTRAIT_DATASET = ""  # Reviewed, source-separated classification dataset directory.
PREPARE_PORTRAIT_DATASET = False
PORTRAIT_REVIEW_CSV = ""  # Reviewed CSV with explicit source-separated split column.
if PREPARE_PORTRAIT_DATASET:
    prepare_classifier_dataset(PORTRAIT_REVIEW_CSV, PORTRAIT_DATASET)
if TRAIN_PORTRAITS:
    from ultralytics import YOLO

    data_root = Path(PORTRAIT_DATASET)
    if not (data_root / "dataset_manifest.json").is_file():
        raise ValueError("Prepare the reviewed dataset and manifest first.")
    classifier = YOLO("yolo11n-cls.pt")
    classifier.train(
        data=str(data_root),
        epochs=30,
        imgsz=224,
        batch=16,
        device=DEVICE,
        amp=True,
        seed=42,
        project=str(WORK_ROOT / "portrait_training"),
        name="characters",
        exist_ok=False,
    )
    print("Use the reported best.pt as PORTRAIT_CLASSIFIER_WEIGHTS, not WEIGHTS.")

In [ ]:
# Optional evaluation after independently labeling saved segment samples.
SEGMENT_REVIEW_LABELS = None  # Copy suggestions to segment_review_labels.csv before editing.
if SEGMENT_REVIEW_LABELS:
    import csv

    with Path(SEGMENT_REVIEW_LABELS).open(newline="", encoding="utf-8") as stream:
        report = evaluate_segments(list(csv.DictReader(stream)))
    print(json.dumps(report, indent=2))
    Path(SEGMENT_REVIEW_LABELS).with_suffix(".evaluation.json").write_text(
        json.dumps(report, indent=2)
    )

# Merge CSV exports copied from independent notebooks, then download the archive.
csv_files = sorted((WORK_ROOT / "jobs").glob("*/observations.csv"))
if not csv_files:
    raise ValueError("No extraction output to export")
merged = merge(csv_files, WORK_ROOT / "merged_observations.csv")
export_rosters(merged, WORK_ROOT / "merged_roster_observations.csv", catalog=ROSTER_CATALOG)
from zipfile import ZipFile, ZIP_DEFLATED

archive = WORK_ROOT.parent / "valomind_results.zip"
# Export metadata/state only; the downloaded video made the first archive 1.6 GB.
INCLUDE_SOURCE_VIDEO = False
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
    for path in WORK_ROOT.rglob("*"):
        if not path.is_file():
            continue
        if "cookie" in path.name.lower() or path.name == ".env":
            continue
        if not INCLUDE_SOURCE_VIDEO and path.suffix.lower() in {".mp4", ".webm", ".mkv"}:
            continue
        bundle.write(path, path.relative_to(WORK_ROOT))
print("Merged CSV:", merged, "\nArchive:", archive)
try:
    from google.colab import files

    files.download(str(archive))
except ImportError:
    print("Kaggle: save the notebook output or download the archive from Files.")